# Moteur de recherche — FIFA World Cup

Projet de Programmation de spécialité (Python), **Partie 3 : interface graphique**.

Conformément aux consignes, **seule l'interface** est écrite dans ce notebook : toutes les
classes (`Document`, `Corpus`, `SearchEngine`…) sont importées depuis les modules `.py` du projet.

**Pour utiliser l'interface** : menu *Run All* (ou exécuter les cellules dans l'ordre).

## 1. Préparation : réutiliser le moteur

### 1.1 Import des classes et chargement du corpus
Le corpus nettoyé est rechargé depuis `data/corpus_sauvegarde.csv` (méthode `load`) :
aucun appel aux API Reddit ou Arxiv.

In [1]:
from pathlib import Path

from Corpus import Corpus

corpus = Corpus("FIFA World Cup")
corpus.load(Path("data") / "corpus_sauvegarde.csv")
corpus

Corpus(nom='FIFA World Cup', 172 documents, 134 auteurs)

### 1.2 et 1.3 Instanciation du moteur
`SearchEngine(corpus)` construit aussitôt le vocabulaire et les matrices TF et TF-IDF.
Les barres de progression `tqdm` (1.3) affichent l'avancement de la construction.

In [2]:
from SearchEngine import SearchEngine

moteur = SearchEngine(corpus)
print(f"{moteur.mat_TF.shape[0]} documents x {moteur.mat_TF.shape[1]} mots")

Nettoyage des textes:   0%|          | 0/172 [00:00<?, ?it/s]

Matrice TF:   0%|          | 0/172 [00:00<?, ?it/s]

172 documents x 3932 mots


Vérification rapide : deux requêtes en mots-clés, comme en Partie 2.

In [3]:
moteur.search("messi argentina", 3)

,id,score,type,titre,auteur,date,url
0,63,0.406,reddit,Lionel Messi retires from Argentina internatio...,TheAthletic,31/08/2026,https://www.reddit.com/r/worldcup/comments/1w3...
1,24,0.160,reddit,Kylian Mbappé reaches Miroslav Klose’s goal re...,Swiftvoyager1906,23/06/2026,https://www.reddit.com/r/worldcup/comments/1ud...
2,58,0.152,reddit,Messi almost catches Vozinha off guard but the...,JCHazard,03/07/2026,https://www.reddit.com/r/worldcup/comments/1um...


In [4]:
moteur.search("cape verde", 3)

,id,score,type,titre,auteur,date,url
0,58,0.499,reddit,Messi almost catches Vozinha off guard but the...,JCHazard,03/07/2026,https://www.reddit.com/r/worldcup/comments/1um...
1,26,0.287,reddit,A journalist was interviewing a Cabo Verde fan...,GiveMeSomeSunshine3,04/07/2026,https://www.reddit.com/r/worldcup/comments/1un...
2,35,0.272,reddit,Post-Match Thread: Argentina 3-2 Cape Verde Is...,matchpal-live,04/07/2026,https://www.reddit.com/r/worldcup/comments/1um...


## 2. Interface graphique avec ipywidgets

### 2.1 Les objets graphiques simples
- un `Label` pour le titre ;
- un `Text` pour saisir les mots-clés ;
- un `IntSlider` (curseur d'entiers) pour choisir le nombre de documents à retourner.

In [5]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

titre = widgets.Label(value="Moteur de recherche")
# Un Label n'accepte pas de HTML : on agrandit le texte par son style
titre.style.font_size = "24px"
titre.style.font_weight = "bold"

champ_mots_cles = widgets.Text(
    description="Mots clés",
    placeholder="ex. : messi argentina",          # texte grisé tant que le champ est vide
    style={"description_width": "initial"},       # l'étiquette n'est pas tronquée
)

curseur_nombre = widgets.IntSlider(
    description="Nombre d'articles à extraire",
    value=5, min=1, max=20,                       # 5 par défaut, entre 1 et 20
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),         # curseur plus large
)

### Bonus 1 : les filtres de recherche
Facultatifs : sans filtre, la recherche porte sur tout le corpus.
- deux `Dropdown` (listes déroulantes) : la source (via `getType`) et l'auteur ;
- deux `DatePicker` (sélecteurs de date) : le début et la fin de la période (bornes incluses) ;
- un bouton pour réinitialiser les filtres.

In [6]:
# Dropdown : chaque option est un couple (texte affiché, valeur renvoyée par .value).
# L'option « pas de filtre » a la valeur "" (chaîne vide) et non None : pour un
# Dropdown, value=None signifie « aucune option sélectionnée » (liste affichée vide).
filtre_source = widgets.Dropdown(
    description="Source",
    options=[("Toutes", ""), ("Reddit", "reddit"), ("Arxiv", "arxiv")],
    value="",
)

# Les auteurs du corpus, par ordre alphabétique (sans tenir compte des majuscules),
# avec leur nombre de documents
auteurs_tries = sorted(corpus.authors.values(), key=lambda auteur: auteur.name.lower())
filtre_auteur = widgets.Dropdown(
    description="Auteur",
    options=[("Tous", "")] + [(str(auteur), auteur.name) for auteur in auteurs_tries],
    value="",
    layout=widgets.Layout(width="400px"),
)

# DatePicker : .value vaut None tant qu'aucune date n'est choisie
filtre_date_debut = widgets.DatePicker(description="Du")
filtre_date_fin = widgets.DatePicker(description="Au")

bouton_reinitialiser = widgets.Button(
    description="Réinitialiser les filtres",
    icon="undo",
    layout=widgets.Layout(width="auto"),      # largeur adaptée au texte
)


def reinitialiser_filtres(bouton: widgets.Button) -> None:
    """Remet tous les filtres sur « pas de filtre »."""
    filtre_source.value = ""
    filtre_auteur.value = ""
    filtre_date_debut.value = None
    filtre_date_fin.value = None


bouton_reinitialiser.on_click(reinitialiser_filtres)

### Bonus 6 : choix de la mesure de pertinence
Des `ToggleButtons` (boutons à bascule : un seul enfoncé à la fois) pour choisir entre la
similarité cosinus sur TF-IDF (Partie 2) et la mesure Okapi BM25.

In [7]:
choix_mesure = widgets.ToggleButtons(
    description="Mesure",
    options=[("TF-IDF (cosinus)", "tfidf"), ("Okapi BM25", "bm25")],
    value="tfidf",
    tooltips=["Similarité cosinus sur les vecteurs TF-IDF (Partie 2)",
              "Okapi BM25 : saturation du TF et normalisation par la longueur"],
)

### Bonus racinisation
Un second moteur est construit avec `racinisation=True` : chaque mot y est remplacé par sa
racine (module `Racinisation`), dans les documents comme dans la requête. Une case à cocher
choisit le moteur utilisé pour la recherche.

In [8]:
moteur_racines = SearchEngine(corpus, racinisation=True)
print(f"Vocabulaire : {len(moteur.vocab)} mots, {len(moteur_racines.vocab)} racines")

case_racinisation = widgets.Checkbox(
    value=False,
    description="Racinisation (predicting, prediction -> predict)",
    style={"description_width": "initial"},
)

Nettoyage des textes:   0%|          | 0/172 [00:00<?, ?it/s]

Matrice TF:   0%|          | 0/172 [00:00<?, ?it/s]

Vocabulaire : 3932 mots, 3085 racines


### 2.3 La zone de résultats
Un `Output` : une zone qui recevra tout ce que la recherche affiche.

In [9]:
zone_resultats = widgets.Output()

### 2.5 La fonction `clique_bouton`
Elle lit les widgets (attribut `.value`), lance la recherche et affiche le résultat dans la
zone `Output` grâce à l'instruction `with`. La zone est vidée à chaque appel
(`clear_output`) : une nouvelle recherche remplace la précédente au lieu de s'y ajouter.
Cela neutralise aussi un comportement observé dans VS Code, qui transmet parfois un même
clic deux fois : le second appel efface l'affichage du premier.
Les cas particuliers (champ vide, aucun résultat) affichent un message.

**Bonus 1** : les filtres choisis sont transmis à `moteur.search` ; une période incohérente (début après la fin) affiche un message d'erreur.

**Bonus « Vouliez-vous dire… ? »** : si des mots de la requête sont inconnus du corpus (faute de frappe), une requête corrigée est proposée (distance de Levenshtein, module `Orthographe`), avec un bouton pour la lancer.

In [10]:
def lien_cliquable(url: str) -> str:
    """Transforme une adresse en lien HTML qui s'ouvre dans un nouvel onglet."""
    return f'<a href="{url}" target="_blank">ouvrir</a>'


def afficher_suggestion(suggestion: str) -> None:
    """Affiche « Vouliez-vous dire… ? » et un bouton qui lance la requête corrigée."""
    bouton_suggestion = widgets.Button(description=f"Rechercher « {suggestion} »",
                                       icon="magic", layout=widgets.Layout(width="auto"))

    def relancer(bouton: widgets.Button) -> None:
        champ_mots_cles.value = suggestion     # le champ affiche la requête corrigée
        clique_bouton(bouton)                  # et on relance la recherche

    bouton_suggestion.on_click(relancer)
    # margin : marge à droite du texte, pour l'écarter du bouton
    texte = widgets.HTML(f"Vouliez-vous dire : <b>{suggestion}</b> ?",
                         layout=widgets.Layout(margin="0 12px 0 0"))
    display(widgets.HBox([texte, bouton_suggestion],
                         layout=widgets.Layout(align_items="center")))


def clique_bouton(bouton: widgets.Button) -> None:
    """Appelée à chaque clic sur « Rechercher » (le bouton est passé en paramètre)."""
    mots_cles = champ_mots_cles.value.strip()      # strip : espaces au bord ignorés
    nombre = curseur_nombre.value

    # Tout ce qui est affiché dans ce bloc « with » va dans la zone de résultats
    with zone_resultats:
        zone_resultats.clear_output()      # on efface la recherche précédente

        if not mots_cles:                  # champ vide (ou seulement des espaces)
            print("Veuillez saisir au moins un mot-clé.")
            return

        # Bonus racinisation : on choisit le moteur selon la case à cocher
        moteur_choisi = moteur_racines if case_racinisation.value else moteur

        # Bonus « Vouliez-vous dire… ? » : mots inconnus -> requête corrigée
        suggestion = moteur_choisi.corriger_requete(mots_cles)
        if suggestion is not None:
            afficher_suggestion(suggestion)
        try:
            resultats = moteur_choisi.search(
                mots_cles, nombre,
                # Bonus 1 : filtres ; « or None » transforme "" (pas de
                # filtre dans le Dropdown) en None (pas de filtre pour search)
                source=filtre_source.value or None,
                auteur=filtre_auteur.value or None,
                date_debut=filtre_date_debut.value,
                date_fin=filtre_date_fin.value,
                mesure=choix_mesure.value,              # Bonus 6 : TF-IDF ou BM25
            )
        except ValueError as erreur:       # ex. : date de début après la date de fin
            print(f"Erreur : {erreur}")
            return

        if resultats.empty:                # aucun document ne correspond
            print(f"Aucun document ne correspond à « {mots_cles} » avec ces filtres.")
            return

        print(f"{len(resultats)} document(s) trouvé(s) pour « {mots_cles} » "
              f"(mesure : {choix_mesure.label}"
              f"{', avec racinisation' if case_racinisation.value else ''}) :")
        # Styler pandas : tableau sans index, score à 3 décimales,
        # URL transformée en lien cliquable
        display(resultats.style
                .format({"url": lien_cliquable, "score": "{:.3f}"})
                .hide(axis="index"))

### 2.4 Le bouton « Rechercher » et son branchement
`on_click` **ajoute** la fonction à la liste de celles que le bouton appelle. Le bouton est
donc créé **dans la même cellule** que son branchement : relancer cette cellule crée un
bouton neuf, branché une seule fois (sinon la fonction serait branchée une fois de plus).

In [11]:
bouton_rechercher = widgets.Button(
    description="Rechercher",
    button_style="primary",      # bouton bleu
    icon="search",               # icône de loupe
)

# On passe la fonction elle-même, SANS parenthèses : c'est le bouton qui l'appellera
bouton_rechercher.on_click(clique_bouton)

### Bonus 8 : concordancier interactif
Réutilise la méthode `concorde` de la classe `Corpus` (Partie 2) : pour chaque occurrence de
l'expression, le contexte gauche, le motif trouvé et le contexte droit. L'expression est une
expression régulière : `penalt(y|ies)` trouve *penalty* et *penalties*.

In [12]:
MAX_LIGNES_CONCORDANCE = 100      # au-delà, on n'affiche que les premières occurrences

champ_expression = widgets.Text(
    description="Expression",
    placeholder="ex. : penalt(y|ies)",
)
curseur_contexte = widgets.IntSlider(
    description="Taille du contexte (caractères)",
    value=35, min=10, max=100, step=5,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)
zone_concordance = widgets.Output()


def clique_concordancier(bouton: widgets.Button) -> None:
    """Affiche le concordancier de l'expression saisie."""
    expression = champ_expression.value.strip()
    with zone_concordance:
        zone_concordance.clear_output()

        if not expression:
            print("Veuillez saisir une expression.")
            return

        try:
            concordance = corpus.concorde(expression, curseur_contexte.value)
        except ValueError as erreur:        # expression régulière invalide, ex. « (goal »
            print(f"Erreur : {erreur}")
            return

        if concordance.empty:
            print(f"Aucune occurrence de « {expression} » dans le corpus.")
            return

        total = len(concordance)
        if total > MAX_LIGNES_CONCORDANCE:
            print(f"{total} occurrences : affichage des {MAX_LIGNES_CONCORDANCE} premières.")
        else:
            print(f"{total} occurrence(s) de « {expression} » :")

        # Alignement façon concordancier : contexte gauche calé à droite, motif
        # centré et en gras, contexte droit calé à gauche
        display(concordance.head(MAX_LIGNES_CONCORDANCE).style
                .set_properties(subset=["contexte gauche"], **{"text-align": "right"})
                .set_properties(subset=["motif trouvé"],
                                **{"text-align": "center", "font-weight": "bold"})
                .set_properties(subset=["contexte droit"], **{"text-align": "left"})
                .hide(axis="index"))


bouton_concordancier = widgets.Button(description="Afficher les concordances",
                                      button_style="primary", icon="list",
                                      layout=widgets.Layout(width="auto"))
bouton_concordancier.on_click(clique_concordancier)

page_concordancier = widgets.VBox([
    widgets.HBox([champ_expression, bouton_concordancier]),
    curseur_contexte,
    zone_concordance,
])

### Bonus 4 : comparaison des sous-corpus Reddit et Arxiv
Chaque sous-corpus est traité comme un gros document (méthode `comparer_sources` de
`SearchEngine`) : mots communs aux deux sources, mots spécifiques à chacune, et un résumé
chiffré. Les mots vides (*the, is, and*…) peuvent être exclus ou non.

**Bonus 7** : sous les tableaux, un diagramme en barres (module `Visualisation`) montre l'importance des mots de chaque liste.

In [13]:
import io

from Visualisation import graphique_sources

curseur_nb_mots = widgets.IntSlider(
    description="Nombre de mots par liste",
    value=15, min=5, max=40, step=5,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)
case_mots_vides = widgets.Checkbox(value=True, description="Exclure les mots vides (the, is, and...)",
                                   style={"description_width": "initial"})
zone_comparaison = widgets.Output()


def tableau_html(titre_tableau: str, tableau: pd.DataFrame) -> widgets.HTML:
    """Un tableau pandas avec son titre, dans un widget HTML (pour les placer côte à côte)."""
    code_html = (tableau.style
                 .hide(axis="index")
                 .format(precision=2)          # 2 décimales au lieu de 6
                 # CSS : espace autour du texte de chaque case (en-têtes th, cellules td)
                 .set_table_styles([{"selector": "th, td", "props": "padding: 3px 12px;"}])
                 .to_html())
    # margin : marge à droite, pour espacer les tableaux placés côte à côte
    return widgets.HTML(f"<b>{titre_tableau}</b>" + code_html,
                        layout=widgets.Layout(margin="0 30px 0 0"))


def figure_en_image(figure) -> widgets.Image:
    """Convertit une figure matplotlib en image PNG affichable dans l'interface.

    Passer par une image fonctionne dans tous les éditeurs (Jupyter, VS Code),
    quel que soit le réglage d'affichage de matplotlib.
    """
    tampon = io.BytesIO()                      # « fichier » en mémoire, pas sur le disque
    figure.savefig(tampon, format="png", dpi=100)
    return widgets.Image(value=tampon.getvalue(), format="png")


def clique_comparaison(bouton: widgets.Button) -> None:
    """Affiche le résumé puis les trois listes de mots, côte à côte."""
    with zone_comparaison:
        zone_comparaison.clear_output()
        resultat = moteur.comparer_sources(curseur_nb_mots.value,
                                           exclure_mots_vides=case_mots_vides.value)
        display(tableau_html("Résumé", resultat["resume"]))
        display(widgets.HBox([
            tableau_html("Mots communs", resultat["communs"]),
            tableau_html("Spécifiques à Reddit", resultat["reddit"]),
            tableau_html("Spécifiques à Arxiv", resultat["arxiv"]),
        ]))
        # Bonus 7 : le même résultat, sous forme de diagrammes en barres
        display(figure_en_image(graphique_sources(resultat, curseur_nb_mots.value)))


bouton_comparaison = widgets.Button(description="Comparer Reddit et Arxiv",
                                    button_style="primary", icon="balance-scale",
                                    layout=widgets.Layout(width="auto"))
bouton_comparaison.on_click(clique_comparaison)

page_comparaison = widgets.VBox([
    widgets.HBox([curseur_nb_mots, bouton_comparaison]),
    case_mots_vides,
    zone_comparaison,
])

### 2.2 Assemblage et affichage de l'interface
`VBox` empile les éléments de haut en bas ; `HBox` (boîte horizontale) place des éléments
côte à côte. L'interface est organisée en **onglets** (`Tab`) : chaque onglet est une
page, et chaque fonctionnalité (recherche, puis les bonus) a la sienne.

In [14]:
# Page de l'onglet « Recherche » : les éléments de la section 2 et les filtres
page_recherche = widgets.VBox([
    widgets.HBox([champ_mots_cles, bouton_rechercher]),
    curseur_nombre,
    choix_mesure,                                          # Bonus 6
    case_racinisation,                                     # Bonus racinisation
    widgets.HTML("<b>Filtres (facultatifs)</b>"),          # Bonus 1
    widgets.HBox([filtre_source, filtre_auteur]),
    widgets.HBox([filtre_date_debut, filtre_date_fin, bouton_reinitialiser]),
    zone_resultats,
])

# Tab : une liste de pages (children) et leurs titres, dans le même ordre
onglets = widgets.Tab(
    children=[page_recherche, page_concordancier, page_comparaison],
    titles=["Recherche", "Concordancier", "Reddit vs Arxiv"],   # Bonus 8, 4
)

interface = widgets.VBox([titre, onglets])     # le titre reste au-dessus des onglets
display(interface)

### 2.6 Vérification

**Comparaison avec la ligne de commande.** La recherche « cape verde » (5 articles) lancée
dans `main_partie2.py` (Partie 2) a donné :

| id | score | titre (début) |
|---|---|---|
| 58 | 0.499 | Messi almost catches Vozinha off guard… |
| 26 | 0.287 | A journalist was interviewing a Cabo Verde fan… |
| 35 | 0.272 | Post-Match Thread: Argentina 3-2 Cape Verde Islands… |
| 66 | 0.254 | Post-Match Thread: Spain 0-0 Cape Verde Islands… |
| 8 | 0.246 | Cape Verde coach Bubista has made it a tradition… |

La même recherche dans l'interface (et dans la cellule ci-dessous) doit donner exactement
le même classement : l'interface ne fait qu'appeler `moteur.search`.

In [15]:
moteur.search("cape verde", 5)[["id", "score", "titre"]]

,id,score,titre
0,58,0.499,Messi almost catches Vozinha off guard but the...
1,26,0.287,A journalist was interviewing a Cabo Verde fan...
2,35,0.272,Post-Match Thread: Argentina 3-2 Cape Verde Is...
3,66,0.254,Post-Match Thread: Spain 0-0 Cape Verde Island...
4,8,0.246,Cape Verde coach Bubista has made it a traditi...


**Majuscules et ponctuation.** La requête est nettoyée comme les documents
(`nettoyer_texte`) : « CAPE Verde !! » doit donner le même résultat que « cape verde ».

In [16]:
moteur.search("CAPE Verde !!", 5).equals(moteur.search("cape verde", 5))

True

**Cas particuliers testés dans l'interface**

| Saisie | Résultat attendu |
|---|---|
| champ vide, ou seulement des espaces | « Veuillez saisir au moins un mot-clé. » |
| mot absent du corpus (`ronaldo`, `cr7 portugal`) | « Aucun document ne contient les mots… » |
| plusieurs clics de suite | la nouvelle recherche remplace la précédente (`clear_output`) |
| curseur au minimum (1) puis au maximum (20) | 1 document, puis au plus 20 (moins si peu de documents contiennent les mots) |
| clic sur un lien « ouvrir » | la page Reddit ou Arxiv d'origine s'ouvre dans un nouvel onglet |

## Bonus 6 : comparaison des classements TF-IDF et Okapi BM25
Pour quelques requêtes, les 5 premiers documents selon chaque mesure, côte à côte
(méthode `comparer_mesures` de `SearchEngine`), et le nombre de documents communs aux
deux listes. Les scores ne sont pas comparables entre eux (un cosinus est compris entre
0 et 1, un score BM25 non) : seuls les classements le sont.

In [17]:
requetes_test = ["messi argentina", "cape verde", "predicting match outcomes",
                 "referee decision"]

for requete in requetes_test:
    comparaison = moteur.comparer_mesures(requete, 5)
    # Documents présents dans les deux listes, quel que soit leur rang
    communs = set(comparaison["id TF-IDF"].dropna()) & set(comparaison["id BM25"].dropna())
    meme_premier = comparaison["id TF-IDF"].iloc[0] == comparaison["id BM25"].iloc[0]
    print(f"« {requete} » : {len(communs)} document(s) commun(s) sur 5, "
          f"même premier document : {'oui' if meme_premier else 'non'}")
    display(comparaison.style.hide(axis="index"))

« messi argentina » : 5 document(s) commun(s) sur 5, même premier document : oui


rang,id TF-IDF,titre TF-IDF,id BM25,titre BM25
1,63,Lionel Messi retires from Argentina international duty with decision ‘that hurts soul deeply’,63,Lionel Messi retires from Argentina international duty with decision ‘that hurts soul deeply’
2,24,Kylian Mbappé reaches Miroslav Klose’s goal record at the World Cup with 16 goals. Closing up to Lionel Messi's record of 18 World Cup goals,58,Messi almost catches Vozinha off guard but the Cape Verde goalkeeper pulls off another great save! 🇨🇻 LFG Cape Verde
3,58,Messi almost catches Vozinha off guard but the Cape Verde goalkeeper pulls off another great save! 🇨🇻 LFG Cape Verde,24,Kylian Mbappé reaches Miroslav Klose’s goal record at the World Cup with 16 goals. Closing up to Lionel Messi's record of 18 World Cup goals
4,22,Messi’s foul against Algeria in 30th minute - No yellow or red card given,22,Messi’s foul against Algeria in 30th minute - No yellow or red card given
5,70,"Leandro Paredes gets 10 match ban for World final brawl, Argentina fined for Falkland Islands banner",70,"Leandro Paredes gets 10 match ban for World final brawl, Argentina fined for Falkland Islands banner"


« cape verde » : 5 document(s) commun(s) sur 5, même premier document : oui


rang,id TF-IDF,titre TF-IDF,id BM25,titre BM25
1,58,Messi almost catches Vozinha off guard but the Cape Verde goalkeeper pulls off another great save! 🇨🇻 LFG Cape Verde,58,Messi almost catches Vozinha off guard but the Cape Verde goalkeeper pulls off another great save! 🇨🇻 LFG Cape Verde
2,26,A journalist was interviewing a Cabo Verde fan outside Miami stadium about Goalkeeper Vozinha’s performance. He turned out to be the father of Vozinha.,26,A journalist was interviewing a Cabo Verde fan outside Miami stadium about Goalkeeper Vozinha’s performance. He turned out to be the father of Vozinha.
3,35,Post-Match Thread: Argentina 3-2 Cape Verde Islands | World Cup | Round of 32,66,Post-Match Thread: Spain 0-0 Cape Verde Islands | World Cup | Group H
4,66,Post-Match Thread: Spain 0-0 Cape Verde Islands | World Cup | Group H,35,Post-Match Thread: Argentina 3-2 Cape Verde Islands | World Cup | Round of 32
5,8,"Cape Verde coach Bubista has made it a tradition to present a pennant to the opposing team's coach before every match. It's great to see this gesture of respect continue even on football's biggest stage, the World Cup.",8,"Cape Verde coach Bubista has made it a tradition to present a pennant to the opposing team's coach before every match. It's great to see this gesture of respect continue even on football's biggest stage, the World Cup."


« predicting match outcomes » : 4 document(s) commun(s) sur 5, même premier document : non


rang,id TF-IDF,titre TF-IDF,id BM25,titre BM25
1,163,FootBots: A Transformer-based Architecture for Motion Prediction in Soccer,160,Machine Learning for Soccer Match Result Prediction
2,160,Machine Learning for Soccer Match Result Prediction,159,AI World Cup 2026: Benchmarking Large Language Models for End-to-End Football Tournament Prediction
3,159,AI World Cup 2026: Benchmarking Large Language Models for End-to-End Football Tournament Prediction,163,FootBots: A Transformer-based Architecture for Motion Prediction in Soccer
4,94,From Players to Champions: A Generalizable Machine Learning Approach for Match Outcome Prediction with Insights from the FIFA World Cup,135,Forecasting Soccer Matches through Distributions
5,122,Forecasting Events in Soccer Matches Through Language,122,Forecasting Events in Soccer Matches Through Language


« referee decision » : 4 document(s) commun(s) sur 5, même premier document : oui


rang,id TF-IDF,titre TF-IDF,id BM25,titre BM25
1,143,SoccerChat: Integrating Multimodal Data for Enhanced Soccer Game Understanding,143,SoccerChat: Integrating Multimodal Data for Enhanced Soccer Game Understanding
2,27,"Argentine fans' petition to replay 2026 World Cup Final passes 80,000 after claims referee 'was bribed'",156,Real Time Offside Detection using a Single Camera in Soccer
3,156,Real Time Offside Detection using a Single Camera in Soccer,27,"Argentine fans' petition to replay 2026 World Cup Final passes 80,000 after claims referee 'was bribed'"
4,63,Lionel Messi retires from Argentina international duty with decision ‘that hurts soul deeply’,63,Lionel Messi retires from Argentina international duty with decision ‘that hurts soul deeply’
5,65,"The more I think about it, the more of a horrendous decision it was to have the World Cup Final at MetLife Stadium",62,"[Mark Ogden] Royal Belgian Football Association (RBFA) is “astonished” by FIFA’s decision to declare Folarin Balogun eligible for USA-Belgium WC R16 game on Monday, RBFA says in statement. RBFA “investigating all potential options.”"
